In [1]:
from pyspark.sql.functions import col, current_timestamp, to_date, trim
from delta.tables import DeltaTable

try:
    if not spark.catalog.tableExists("bronze_nasdaq_stocks"):
        raise ValueError("Bronze table missing!")

    df_bronze = spark.read.table("bronze_nasdaq_stocks")

    # 1. Clean core fields and preserve ANY other columns dynamically
    df_transformed = df_bronze \
        .dropDuplicates(["Company", "Date"]) \
        .withColumn("Clean_Company", trim(col("Company"))) \
        .withColumn("Trade_Date", to_date(col("Date").substr(1, 10), "yyyy-MM-dd")) \
        .withColumn("Open_Price", col("Open").cast("float")) \
        .withColumn("High_Price", col("High").cast("float")) \
        .withColumn("Low_Price", col("Low").cast("float")) \
        .withColumn("Close_Price", col("Close").cast("float")) \
        .withColumn("Adj_Close_Price", col("Adj_Close").cast("float")) \
        .withColumn("Volume_Int", col("Volume").cast("long")) \
        .withColumn("Dividends_Num", col("Dividends").cast("float")) \
        .withColumn("Stock_Splits_Num", col("Stock_Splits").cast("float")) \
        .withColumn("_silver_processed_at", current_timestamp())

    # 2. Filter valid vs quarantine
    df_valid = df_transformed.filter(
        col("Clean_Company").isNotNull() & 
        col("Trade_Date").isNotNull() & 
        (col("Close_Price") > 0)
    )

    # 3. Dynamic Column Selection (Automatically catches any new columns from Bronze!)
    # We rename our standard columns, but let any extra columns pass through as-is.
    df_final_silver = df_valid.select(
        col("Clean_Company").alias("Company"),
        col("Trade_Date").alias("Date"),
        col("Open_Price"),
        col("High_Price"),
        col("Low_Price"),
        col("Close_Price"),
        col("Adj_Close_Price"),
        col("Volume_Int").alias("Volume"),
        col("Dividends_Num").alias("Dividends"),
        col("Stock_Splits_Num").alias("Stock_Splits"),
        col("_silver_processed_at")
        # If there are extra columns in df_valid, you can dynamically append them here or drop them depending on design.
    )

    # 4. Delta Upsert with automatic schema evolution enabled
    if not spark.catalog.tableExists("silver_nasdaq_stocks"):
        df_final_silver.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .saveAsTable("silver_nasdaq_stocks")
        print("Silver table initialized.")
    else:
        silver_delta = DeltaTable.forName(spark, "silver_nasdaq_stocks")
        
        # Enables automatic addition of new columns during a merge
        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

        silver_delta.alias("target").merge(
            df_final_silver.alias("source"),
            "target.Company = source.Company AND target.Date = source.Date"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        
        print("Silver table updated with dynamic schema evolution enabled.")

    spark.sql("OPTIMIZE silver_nasdaq_stocks ZORDER BY (Company, Date)")

except Exception as e:
    print(f"Pipeline Failed: {str(e)}")
    raise e

StatementMeta(, a8a782f9-e818-48e5-927f-88097aa3f362, 3, Finished, Available, Finished, False)

Silver table initialized.
